{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Clean WDM Retrieval Notebook\n",
    "\n",
    "This notebook is a thin Jupyter wrapper around `wdm_retrieval_clean.py`.\n",
    "\n",
    "It keeps the retrieval logic clean and avoids the earlier `numpy.ndarray` issue with `wdm.csvtowdm` by ensuring all WDM inputs remain pandas objects with a `DatetimeIndex`.\n",
    "\n",
    "**Before running this notebook:** make sure your Earthdata credentials and token setup are already working in your environment."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pathlib import Path\n",
    "import os\n",
    "import pandas as pd\n",
    "\n",
    "from wdm_retrieval_clean import (\n",
    "    CONSTITUENTS,\n",
    "    get_catalog,\n",
    "    load_stations,\n",
    "    run_retrieval,\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Edit these parameters\n",
    "\n",
    "You can point `station_csv` to your own station list, or keep the provided example file."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "workspace = Path.cwd()\n",
    "station_csv = workspace / \"station_example.csv\"\n",
    "start_date = \"2025-01-01\"\n",
    "end_date = \"2025-01-03\"\n",
    "wdm_file = workspace / \"MetData_clean_notebook.wdm\"\n",
    "log_file = workspace / \"MetLog_clean_notebook.txt\"\n",
    "\n",
    "print(f\"Workspace: {workspace}\")\n",
    "print(f\"Station CSV: {station_csv}\")\n",
    "print(f\"Date range: {start_date} to {end_date}\")\n",
    "print(f\"WDM output: {wdm_file}\")\n",
    "print(f\"Log output: {log_file}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Preview stations and dataset selection\n",
    "\n",
    "This cell validates the station file and shows whether each station will use NLDAS or GLDAS."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "stations = load_stations(station_csv)\n",
    "print(f\"Loaded {len(stations)} station(s).\")\n",
    "\n",
    "preview_rows = []\n",
    "for station in stations:\n",
    "    catalog, timestep_hours, dataset_name = get_catalog(station)\n",
    "    preview_rows.append({\n",
    "        \"StationName\": station.name,\n",
    "        \"Latitude\": station.latitude,\n",
    "        \"Longitude\": station.longitude,\n",
    "        \"TimeZoneAdjustment\": station.timezone_adjustment,\n",
    "        \"Dataset\": dataset_name,\n",
    "        \"TimeStepHours\": timestep_hours,\n",
    "    })\n",
    "\n",
    "pd.DataFrame(preview_rows)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Constituents to be written\n",
    "\n",
    "These are the constituents handled by the clean workflow."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "CONSTITUENTS"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Run retrieval and write the WDM file\n",
    "\n",
    "This cell will call the clean retrieval pipeline, build the WDM file, and create the MetLog text file."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "created_wdm, created_log = run_retrieval(\n",
    "    stations=stations,\n",
    "    start_date=start_date,\n",
    "    end_date=end_date,\n",
    "    wdm_path=wdm_file,\n",
    "    log_path=log_file,\n",
    ")\n",
    "\n",
    "print(f\"Created WDM file: {created_wdm}\")\n",
    "print(f\"Created log file: {created_log}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Quick output check"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"WDM exists:\", Path(created_wdm).exists())\n",
    "print(\"Log exists:\", Path(created_log).exists())\n",
    "\n",
    "if Path(created_log).exists():\n",
    "    with open(created_log, \"r\", encoding=\"utf-8\") as f:\n",
    "        log_preview = f.readlines()[:15]\n",
    "    print(\"\".join(log_preview))"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

